In [2]:
import requests #fetches data from webpage
from bs4 import BeautifulSoup #extracts data we need
import pandas as pd #to store data as csv or excel format
from datetime import date

WRITE_FILE_PATH = r'c:\Interest_rate_files'
URL = "https://bankofindia.co.in/interest-rate/rupee-term-deposit-rate"

date1 = (date.today()).strftime('%d%m%Y') # Get today's date and convert it to to dd/mm/yy format
file_name= WRITE_FILE_PATH + '\Interest_rate_' + date1 + '.csv' # Set directory path to write file

r=requests.get(URL) # Fetch the HTML page
#print(r)
soup=BeautifulSoup(r.text,'html.parser')
table=soup.find("table",class_="table")
rows=table.find_all("tr")

print("Today's date =",date1)
print("Interest rate file name = ",file_name)

list_of_interest_rates = []
tax_saver_row = []
row_count=0
for i in rows[1:]:  # Skip heading
    data = i.find_all("td")
    row=['Bank of India']
    row.extend([tr.text for tr in data[:2]]) # Extract the text from the first two columns only
    if row_count <= 4: #for the first 5 rows, repeat general rate as Sr citizen rate
        new_row_items = ['',row[2] + '%',''] 
    elif row_count<=12: # for 6th to 12th row (6mths to 3yrs), add 0.5% to Sr citizen rate
        new_row_items = ['',str(float(row[2])+0.50) + '%',''] 
    else: # beyond 12th row (more than 3yrs), add 0.75% to Sr citizen rate
        new_row_items = ['',str(float(row[2])+0.75) + '%','']
    row.extend(new_row_items)
    row[2] = row[2] + '%' # Add % sign to general rate of interest
    if row_count == 14:
            tax_saver_row = ['Bank of India', 'Tax-saver Fixed Deposit', row[2], '', row[4], '']
            list_of_interest_rates.append(tax_saver_row)
    list_of_interest_rates.append(row)
    row_count+=1
           
pd.set_option('display.max_columns',None)
int_rate_df=pd.DataFrame(list_of_interest_rates,columns=['bank_name','tenure','gen_rate','annualised_gen_rate',
                                                        'sr_rate','annualised_sr_rate'])
#print(list_of_interest_rates)
#print(int_rate_df)
int_rate_df.to_csv(file_name,mode='a',header=False,index=False) # Write the interest rate rows to the file
print("\nBank of India = Success. Number of rows added = ", len(list_of_interest_rates))

Today's date = 21112024
Interest rate file name =  c:\Interest_rate_files\Interest_rate_21112024.csv
        bank_name                                             tenure gen_rate  \
0   Bank of India                                  7 days to 14 days    3.00%   
1   Bank of India                                 15 days to 30 days    3.00%   
2   Bank of India                                 31 days to 45 days    3.00%   
3   Bank of India                                 46 days to 90 days    4.50%   
4   Bank of India                                91 days to 179 days    4.50%   
5   Bank of India                               180 days to 210 days    6.00%   
6   Bank of India                               211 days to 269 days    6.00%   
7   Bank of India                       270 days to less than 1 Year    6.00%   
8   Bank of India                                             1 Year    6.80%   
9   Bank of India  Above 1 Year to less than 2 Years (except 400 ...    6.80%   
10  Bank